# TRELLIS Avatar 3D (Kaggle Ready)
Génère un modèle 3D texturé (`.glb`) depuis une image personnage.

## Prérequis Kaggle
- `Settings -> Accelerator -> GPU`
- `Settings -> Internet -> ON`
- Ajouter un dataset contenant `avatar-man-1.png`

In [ ]:
!nvidia-smi

In [ ]:
!cd /kaggle/working && rm -rf TRELLIS && git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git

In [ ]:
!cd /kaggle/working/TRELLIS && . ./setup.sh --demo --xformers --diffoctreerast --spconv --mipgaussian --kaolin --nvdiffrast || true
!python3 -m pip install --upgrade pip
!python3 -m pip install imageio pillow numpy

In [ ]:
import os, json, shutil
import numpy as np
import imageio
from PIL import Image

os.chdir('/kaggle/working/TRELLIS')
os.environ['SPCONV_ALGO'] = 'native'

from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils, render_utils

INPUT_IMAGE = '/kaggle/input/avatar-man/avatar-man-1.png'  # <-- adapte ce chemin
OUT_DIR = '/kaggle/working/outputs/avatar-man-1'
TEXTURE_SIZE = 2048
SIMPLIFY = 0.92
SEEDS = [11, 23, 37]

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f'{OUT_DIR}/variants', exist_ok=True)

img = Image.open(INPUT_IMAGE).convert('RGB')
w, h = img.size
long_side = max(w, h)
if long_side < 1536:
    s = 1536 / long_side
    img = img.resize((int(w*s), int(h*s)), Image.Resampling.LANCZOS)

ref = np.asarray(img)

pipe = TrellisImageTo3DPipeline.from_pretrained('microsoft/TRELLIS-image-large')
pipe.cuda()

def similarity(a, b):
    H = min(a.shape[0], b.shape[0])
    W = min(a.shape[1], b.shape[1])
    a2 = np.asarray(Image.fromarray(a).resize((W, H), Image.Resampling.BILINEAR)).astype(np.float32)/255.
    b2 = np.asarray(Image.fromarray(b).resize((W, H), Image.Resampling.BILINEAR)).astype(np.float32)/255.
    mse = np.mean((a2-b2)**2)
    return 1.0/(1.0+float(mse))

results = []
for seed in SEEDS:
    d = f'{OUT_DIR}/variants/seed_{seed}'
    os.makedirs(d, exist_ok=True)
    out = pipe.run(
        img,
        seed=seed,
        formats=['gaussian', 'mesh'],
        preprocess_image=False,
        sparse_structure_sampler_params={'steps': 12, 'cfg_strength': 7.5},
        slat_sampler_params={'steps': 12, 'cfg_strength': 3.0},
    )
    gs = out['gaussian'][0]
    mesh = out['mesh'][0]
    glb = postprocessing_utils.to_glb(gs, mesh, simplify=SIMPLIFY, texture_size=TEXTURE_SIZE, verbose=False)
    glb_path = f'{d}/seed_{seed}.glb'
    glb.export(glb_path)
    color_video = render_utils.render_video(gs, num_frames=120)['color']
    normal_video = render_utils.render_video(mesh, num_frames=120)['normal']
    color_path = f'{d}/seed_{seed}_color.mp4'
    normal_path = f'{d}/seed_{seed}_turntable.mp4'
    imageio.mimsave(color_path, color_video, fps=24)
    imageio.mimsave(normal_path, normal_video, fps=24)
    score = similarity(ref, np.asarray(color_video[0]))
    results.append({'seed': seed, 'glb': glb_path, 'turntable': normal_path, 'turntable_color': color_path, 'score_combined': score})

best = sorted(results, key=lambda x: x['score_combined'], reverse=True)[0]
shutil.copy2(best['glb'], f'{OUT_DIR}/avatar-man-1_final_raw.glb')
shutil.copy2(best['turntable'], f'{OUT_DIR}/avatar-man-1_turntable_raw.mp4')

manifest = {
    'input_image': INPUT_IMAGE,
    'texture_size': TEXTURE_SIZE,
    'simplify': SIMPLIFY,
    'seeds': SEEDS,
    'selected_seed': best['seed'],
    'selected_score_combined': best['score_combined'],
    'final_raw_glb': f'{OUT_DIR}/avatar-man-1_final_raw.glb',
    'final_raw_turntable': f'{OUT_DIR}/avatar-man-1_turntable_raw.mp4',
    'variants': results,
}
with open(f'{OUT_DIR}/manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

# bundle zip simple
delivery = f'{OUT_DIR}/avatar-man-1_delivery'
os.makedirs(delivery, exist_ok=True)
shutil.copy2(f'{OUT_DIR}/manifest.json', f'{delivery}/manifest.json')
shutil.copy2(f'{OUT_DIR}/avatar-man-1_final_raw.glb', f'{delivery}/avatar-man-1_final_raw.glb')
shutil.copy2(f'{OUT_DIR}/avatar-man-1_turntable_raw.mp4', f'{delivery}/avatar-man-1_turntable_raw.mp4')
with open(f'{delivery}/README.txt', 'w') as f:
    f.write('Delivery generated on Kaggle.\n')
shutil.make_archive(f'{OUT_DIR}/avatar-man-1_delivery', 'zip', root_dir=delivery)

print('Done:', best)
print('ZIP:', f'{OUT_DIR}/avatar-man-1_delivery.zip')

In [ ]:
!ls -lah /kaggle/working/outputs/avatar-man-1
!ls -lah /kaggle/working/outputs/avatar-man-1/variants
!ls -lah /kaggle/working/outputs/avatar-man-1/avatar-man-1_delivery
!ls -lah /kaggle/working/outputs/avatar-man-1/avatar-man-1_delivery.zip